# 3. Training

Train and compare all candidate models: Linear Regression, Random Forest, Gradient Boosting, and XGBoost.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from bsdf.config import get_config
from bsdf.data import load_processed_data, split_time_ordered_data
from bsdf.evaluation import compare_models
from bsdf.features import add_time_features, create_feature_target_split
from bsdf.modeling import build_candidate_models, train_models
from bsdf.utils import save_model, save_table

CONFIG = get_config(PROJECT_ROOT)

In [2]:
data = add_time_features(load_processed_data(CONFIG))
train_data, test_data = split_time_ordered_data(data, CONFIG.test_fraction)
X_train, y_train = create_feature_target_split(train_data)
X_test, y_test = create_feature_target_split(test_data)

candidate_models = build_candidate_models(CONFIG.random_state)
trained_models = train_models(candidate_models, X_train, y_train)
comparison = compare_models(trained_models, X_test, y_test)
comparison

,model,mean_absolute_deviation,mean_absolute_percentage_error,r2_score
0,xgboost,50.905449,0.490943,0.886153
1,random_forest,53.471518,0.354383,0.868960
2,gradient_boosting,57.002632,0.608443,0.860422
3,linear_regression,98.690537,2.113512,0.631634


In [3]:
for model_name, model in trained_models.items():
    save_model(model, CONFIG.model_dir / f"{model_name}.joblib")

best_model_name = comparison.iloc[0]["model"]
save_model(trained_models[best_model_name], CONFIG.model_dir / "best_model.joblib")
save_table(comparison, CONFIG.processed_dir / "model_comparison.csv")

print(f"Best model: {best_model_name}")
print("Saved model artifacts to models/ and comparison table to data/processed.")

Best model: xgboost
Saved model artifacts to models/ and comparison table to data/processed.


## Business Model Choice

The recommended production model is selected from all trained candidates using the lowest mean absolute deviation. In the executed workflow, XGBoost is the best model. It is suitable for the business case because it captures nonlinear interactions among hour, working day, season, and weather, while remaining practical to retrain and package as one pipeline artifact.